# Train PointNu-Net on Kaggle PanNuke

Notebook này hướng dẫn chạy train PointNu-Net trên Kaggle theo đúng kiểu notebook: clone source trong notebook, cài dependencies, map dataset PanNuke từ `/kaggle/input`, rồi chạy `train_pannuke.py`.

Notebook được viết để phù hợp với cấu trúc dataset kiểu bạn chụp: `PanNuke/fold_1/Fold 1/...`, nhưng vẫn có logic tìm file đủ linh hoạt để tự nhận diện các fold khác.

In [47]:
from pathlib import Path
import os
import subprocess
import sys

KAGGLE_INPUT = Path('/kaggle/input/datasets/phihungcrr1701')
WORK_DIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/phihungcr1701/PointNu-Net'
REPO_DIR = WORK_DIR / 'PointNu-Net'
DATASET_ROOT = KAGGLE_INPUT / 'pannuke'

print('Kaggle input exists:', KAGGLE_INPUT.exists())
print('Dataset root:', DATASET_ROOT)
print('Dataset root exists:', DATASET_ROOT.exists())
print('Working dir:', WORK_DIR)
print('Repo dir:', REPO_DIR)

Kaggle input exists: True
Dataset root: /kaggle/input/datasets/phihungcrr1701/pannuke
Dataset root exists: True
Working dir: /kaggle/working
Repo dir: /kaggle/working/PointNu-Net


In [45]:
import shutil

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

In [46]:
os.chdir("/kaggle/working")

In [48]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Current directory:', os.getcwd())

# Kaggle Python 3.12 often fails on this repo's pinned legacy requirements.
# Install only packages that are missing in the current environment, using modern package names.
import importlib.util as importlib_util

module_to_package = {
    'albumentations': 'albumentations',
    'imgaug': 'imgaug',
    'yaml': 'PyYAML',
    'tifffile': 'tifffile',
    'cv2': 'opencv-python-headless',
    'skimage': 'scikit-image',
    'timm': 'timm',
    'torchsummaryX': 'torchsummaryX',
}

missing_packages = []
for module_name, package_name in module_to_package.items():
    if importlib_util.find_spec(module_name) is None:
        missing_packages.append(package_name)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing_packages, check=True)
else:
    print('All required packages are already available')

print('Dependencies ready for PanNuke training')

Cloning into '/kaggle/working/PointNu-Net'...


Current directory: /kaggle/working/PointNu-Net
All required packages are already available
Dependencies ready for PanNuke training


In [53]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 396 bytes | 396.00 KiB/s, done.
From https://github.com/phihungcr1701/PointNu-Net
   fe307cb..4e4336b  master     -> origin/master
Updating fe307cb..4e4336b
Fast-forward
 utils/dataloader.py | 5 +++++
 1 file changed, 5 insertions(+)


In [49]:
source_root = DATASET_ROOT
print('Using PanNuke root directly:', source_root)
print('Checking expected Kaggle structure:')

# Exact structure based on your screenshot:
# /kaggle/input/PanNuke/fold_1/Fold 1/images/fold1/images.npy
# /kaggle/input/PanNuke/fold_1/Fold 1/masks/fold1/masks.npy
for fold in [1, 2, 3]:
    expected_img = source_root / f'fold_{fold}' / f'Fold {fold}' / 'images' / f'fold{fold}' / 'images.npy'
    expected_mask = source_root / f'fold_{fold}' / f'Fold {fold}' / 'masks' / f'fold{fold}' / 'masks.npy'
    print(f'Fold {fold}:')
    print('  expected image:', expected_img)
    print('  expected mask :', expected_mask)
    print('  image exists  :', expected_img.exists())
    print('  mask exists   :', expected_mask.exists())

print('No copy/symlink step is needed now.')

Using PanNuke root directly: /kaggle/input/datasets/phihungcrr1701/pannuke
Checking expected Kaggle structure:
Fold 1:
  expected image: /kaggle/input/datasets/phihungcrr1701/pannuke/fold_1/Fold 1/images/fold1/images.npy
  expected mask : /kaggle/input/datasets/phihungcrr1701/pannuke/fold_1/Fold 1/masks/fold1/masks.npy
  image exists  : True
  mask exists   : True
Fold 2:
  expected image: /kaggle/input/datasets/phihungcrr1701/pannuke/fold_2/Fold 2/images/fold2/images.npy
  expected mask : /kaggle/input/datasets/phihungcrr1701/pannuke/fold_2/Fold 2/masks/fold2/masks.npy
  image exists  : True
  mask exists   : True
Fold 3:
  expected image: /kaggle/input/datasets/phihungcrr1701/pannuke/fold_3/Fold 3/images/fold3/images.npy
  expected mask : /kaggle/input/datasets/phihungcrr1701/pannuke/fold_3/Fold 3/masks/fold3/masks.npy
  image exists  : True
  mask exists   : True
No copy/symlink step is needed now.


In [50]:
os.environ['HRNET_W64_PRETRAINED_PATH'] = '/kaggle/input/datasets/phihungcr1701/hrnetv2/hrnetv2_w64_imagenet_pretrained.pth'

In [ ]:
# Chay train voi cau hinh mac dinh cho PanNuke
# Neu bi OOM, hay mo configs/pannuke.yaml va giam train.batch_size truoc khi chay cell nay.
train_env = os.environ.copy()
train_env['CUDA_VISIBLE_DEVICES'] = '0'
print('CUDA_VISIBLE_DEVICES for train =', train_env['CUDA_VISIBLE_DEVICES'])
subprocess.run([
    sys.executable,
    'train_pannuke.py',
    '--name=kaggle_pannuke',
    '--seed=888',
    '--train_fold=1',
    '--val_fold=2',
    '--test_fold=3'
], check=True, env=train_env)

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Using manual seed: 888
processed data with size 2656
Using train augmentation
multi GPU training detection
Freezing backbone stage -1
active train conv1 and bn1
Multi step scheduler decay at 80 and 90 with gamma 0.1


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Traceback (most recent call last):
  File "/kaggle/working/PointNu-Net/train_pannuke.py", line 60, in <module>
    ins_loss, cate_loss,maskiou_loss=trainer.seg_updata(train_data)
                                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/PointNu-Net/trainer.py", line 318, in seg_updata
    feature_preds, kernel_preds, cate_preds = self.model(image)
                                              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/di

CalledProcessError: Command '['/usr/bin/python3', 'train_pannuke.py', '--name=kaggle_pannuke', '--seed=888', '--train_fold=1', '--val_fold=2', '--test_fold=3']' returned non-zero exit status 1.

## Sau khi train

Checkpoint và log sẽ được lưu trong `outputs/kaggle_pannuke/`. Nếu bạn muốn thử fold khác, chỉ cần đổi ba tham số `--train_fold`, `--val_fold`, `--test_fold` theo một hoán vị của 1, 2, 3.

Nếu Kaggle báo thiếu bộ nhớ, cách sửa nhanh nhất là giảm `train.batch_size` trong `configs/pannuke.yaml` xuống `4` hoặc `2`.